# Ingest — GitHub Copilot, from the emailed CSV

Reads the AI usage report from `Files/landing/github/` into `github_ai_usage` and
`github_user_map`.

## Prefer `Ingest_GitHub_API.ipynb` unless you need these three columns

GitHub is the one source that can run **unattended** — the dedicated AI-credit endpoints carry 24
months of history against this report's 31 days, and nobody has to download an email attachment.

Use this CSV version only if you need `total_monthly_quota`, `aic_quantity` or `aic_gross_amount`,
which appear in the export but are not in the API schema. Consumption Central does not depend on any of
them — the pooled allowance is computed from the seat list — so for most people the API notebook is
simply better.

Where it does win: `repository` is in the CSV and not in the AI-credit API response, so if you want
to attribute credits to repositories, this is the only route.

## Getting the file

GitHub → your enterprise → **Billing & Licensing** → Usage → **AI usage** → **Get usage report** →
up to **31 days** → **Email me the report**. The link expires after 24 hours.

Because of the 31-day cap, monthly is the natural cadence and the merge below is built for it.

In [ ]:
LANDING = "Files/landing/github"
TBL_USAGE = "github_ai_usage"
TBL_SEATS = "github_user_map"

# Only used when the seat file omits included_credits. These are GitHub's
# published promotional allowances, which fall to 1,900 / 3,900 on 1 Sep 2026.
DEFAULT_INCLUDED = {"Copilot Business": 3000, "Copilot Enterprise": 7000}

In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
import re


def norm(name):
    return re.sub(r"[ _\-]", "", name).lower()


def pick(df, *aliases):
    lookup = {norm(c): c for c in df.columns}
    for a in aliases:
        if norm(a) in lookup:
            return lookup[norm(a)]
    return None


def col_or_null(df, name, cast):
    """The column if present, otherwise a typed null.

    total_monthly_quota, aic_quantity and aic_gross_amount are in real exports
    but absent from GitHub's published field reference, so they may or may not
    be there depending on when the report was generated.
    """
    return F.col(name).cast(cast) if name else F.lit(None).cast(cast)


def read(pattern):
    try:
        df = (spark.read.option("header", True).option("inferSchema", False)
              .csv(f"{LANDING}/{pattern}"))
        return df if df.count() else None
    except Exception as e:
        print(f"  {pattern}: not found ({type(e).__name__})")
        return None

## Usage

In [ ]:
raw = read("*[Uu]sage*.csv")
if raw is None:
    print("no AI usage export found - skipping")
else:
    print(f"read {raw.count():,} rows")
    print("columns:", raw.columns)

    c_quota = pick(raw, "total_monthly_quota")
    c_aicq = pick(raw, "aic_quantity")
    c_aicg = pick(raw, "aic_gross_amount")
    for name, c in (("total_monthly_quota", c_quota), ("aic_quantity", c_aicq),
                    ("aic_gross_amount", c_aicg)):
        if not c:
            print(f"  note: {name} absent - it is undocumented and not always present")

    usage = raw.select(
        F.to_date(F.col(pick(raw, "date", "usage_date"))).alias("usage_date"),
        F.lower(F.trim(F.col(pick(raw, "username")))).alias("username"),
        F.col(pick(raw, "product")).cast("string").alias("product"),
        F.col(pick(raw, "sku")).cast("string").alias("sku"),
        F.col(pick(raw, "model")).cast("string").alias("model"),
        F.col(pick(raw, "quantity")).cast("double").alias("quantity"),
        F.col(pick(raw, "unit_type")).cast("string").alias("unit_type"),
        F.col(pick(raw, "applied_cost_per_quantity")).cast("double")
            .alias("applied_cost_per_quantity"),
        F.col(pick(raw, "gross_amount")).cast("double").alias("gross_amount"),
        F.col(pick(raw, "discount_amount")).cast("double").alias("discount_amount"),
        F.col(pick(raw, "net_amount")).cast("double").alias("net_amount"),
        col_or_null(raw, c_quota, "double").alias("total_monthly_quota"),
        F.col(pick(raw, "organization")).cast("string").alias("organization"),
        F.col(pick(raw, "repository")).cast("string").alias("repository"),
        F.col(pick(raw, "cost_center_name")).cast("string").alias("cost_center_name"),
    ).withColumn("_loaded_at", F.current_timestamp())

    print(f"  {usage.count():,} rows, "
          f"{usage.select('username').distinct().count():,} developers, "
          f"{usage.select('model').distinct().count()} models")

    # A developer can use several models and SKUs on the same day, and the same
    # model against different repositories, so all five belong in the key.
    KEY = ["usage_date", "username", "sku", "model", "repository"]
    if spark.catalog.tableExists(TBL_USAGE):
        before = spark.table(TBL_USAGE).count()
        cond = " AND ".join(f"t.{k} <=> s.{k}" for k in KEY)
        (DeltaTable.forName(spark, TBL_USAGE).alias("t")
            .merge(usage.alias("s"), cond)
            .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
        after = spark.table(TBL_USAGE).count()
        print(f"  {TBL_USAGE}: {before:,} -> {after:,}  (+{after - before:,})")
    else:
        usage.write.format("delta").saveAsTable(TBL_USAGE)
        print(f"  {TBL_USAGE}: created with {usage.count():,} rows")

## Seats

Current state, so this is a full replace rather than a merge — someone who moved from Business to
Enterprise should appear once, on the plan they are on now.

In [ ]:
raw = read("*[Mm]ap*.csv")
if raw is None:
    print("no seat map found - GitHub cost pages need this for seat prices")
else:
    c_inc = pick(raw, "included_credits")
    seats = raw.select(
        F.lower(F.trim(F.col(pick(raw, "username")))).alias("username"),
        F.lower(F.trim(F.col(pick(raw, "userPrincipalName", "upn", "email"))))
            .alias("user_principal_name"),
        F.col(pick(raw, "displayName", "name")).cast("string").alias("display_name"),
        F.trim(F.col(pick(raw, "plan"))).alias("plan"),
        col_or_null(raw, c_inc, "long").alias("included_credits"),
    )

    if not c_inc:
        print("  included_credits absent - filling from the published allowances")
        seats = seats.withColumn(
            "included_credits",
            F.when(F.col("plan") == "Copilot Enterprise",
                   F.lit(DEFAULT_INCLUDED["Copilot Enterprise"]))
             .otherwise(F.lit(DEFAULT_INCLUDED["Copilot Business"])).cast("long"))

    seats = (seats.dropDuplicates(["username"])
             .withColumn("_loaded_at", F.current_timestamp()))

    # The seat-price measure keys on the plan name, so an unexpected spelling
    # silently prices everyone as Business.
    odd = seats.filter(~F.col("plan").isin("Copilot Business", "Copilot Enterprise"))
    if odd.count():
        print(f"  ! {odd.count()} seats have an unrecognised plan name:")
        odd.select("plan").distinct().show(truncate=False)
        print("    expected exactly 'Copilot Business' or 'Copilot Enterprise'")

    no_upn = seats.filter(F.col("user_principal_name").isNull()).count()
    if no_upn:
        print(f"  ! {no_upn} seats have no UPN - those developers will not appear")
        print("    in any department breakdown")

    seats.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true").saveAsTable(TBL_SEATS)
    print(f"  {TBL_SEATS}: {seats.count():,} seats")
    seats.groupBy("plan").count().show(truncate=False)

## Check

In [ ]:
if spark.catalog.tableExists(TBL_USAGE):
    spark.sql(f"""
        SELECT  MIN(usage_date) AS earliest, MAX(usage_date) AS latest,
                COUNT(DISTINCT usage_date)  AS days,
                COUNT(DISTINCT username)    AS developers,
                ROUND(SUM(gross_amount), 2) AS gross,
                ROUND(SUM(net_amount), 2)   AS net_billable
        FROM    {TBL_USAGE}
    """).show(truncate=False)

    # net well below gross means the pooled allowance is absorbing consumption,
    # which is the normal and healthy state. net near gross means you are paying
    # for most of it.
    spark.sql(f"""
        SELECT  model,
                COUNT(DISTINCT username)    AS developers,
                ROUND(SUM(gross_amount), 2) AS gross,
                ROUND(SUM(net_amount), 2)   AS net
        FROM    {TBL_USAGE}
        GROUP BY model
        ORDER BY gross DESC
    """).show(truncate=False)